<a href="https://colab.research.google.com/github/williamfaraday123/SC4001-Neural-Network/blob/main/Lim_Isaac_Part_B_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question B2 (10 marks)
In Question B1, we used the Category Embedding model. This creates a feedforward neural network in which the categorical features get learnable embeddings. In this question, we will make use of a library called Pytorch-WideDeep. This library makes it easy to work with multimodal deep-learning problems combining images, text, and tables. We will just be utilizing the deeptabular component of this library through the TabMlp network:

In [1]:
!pip install pytorch-widedeep

In [2]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep import Trainer
from pytorch_widedeep.metrics import R2Score

import torch
from sklearn.metrics import mean_squared_error, r2_score

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


1.Divide the dataset (‘hdb_price_prediction.csv’) into train and test sets by using entries from the year 2020 and before as training data, and entries from 2021 and after as the test data.

In [3]:
df = pd.read_csv('hdb_price_prediction.csv')

# YOUR CODE HERE
# Force 'year' to be numeric just in case it's stored as a string
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Splitting the data
train = df[df['year'] < 2020].copy()
val = df[df['year'] == 2020].copy()
test = df[df['year'] >= 2021].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


2.Refer to the documentation of Pytorch-WideDeep and perform the following tasks:
https://pytorch-widedeep.readthedocs.io/en/latest/index.html
* Use [**TabPreprocessor**](https://pytorch-widedeep.readthedocs.io/en/latest/examples/01_preprocessors_and_utils.html#2-tabpreprocessor) to create the deeptabular component using the continuous
features and the categorical features. Use this component to transform the training dataset.
* Create the [**TabMlp**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/model_components.html#pytorch_widedeep.models.tabular.mlp.tab_mlp.TabMlp) model with 2 linear layers in the MLP, with 250 and 150 neurons respectively.
* Create a [**Trainer**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/trainer.html#pytorch_widedeep.training.Trainer) for the training of the created TabMlp model with the mean squared error (MSE) cost function. Train the model for 80 epochs using this trainer, keeping a batch size of 64. (Note: set the *num_workers* parameter to 0.)

In [4]:
# YOUR CODE & RESULT HERE
# Define columns as per previous task
num_cols = [
    "dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality",
    "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"
]
cat_cols = ["month", "town", "flat_model_type", "storey_range"]
target_col = "resale_price"

# Initialize Preprocessor
tab_preprocessor = TabPreprocessor(
    cat_embed_cols=cat_cols,
    continuous_cols=num_cols
)

# Transform the training set
X_tab_train = tab_preprocessor.fit_transform(train)
y_train = train[target_col].values

# Transform the training set
X_tab_val = tab_preprocessor.fit_transform(val)
y_val = val[target_col].values

# Transform test set for later use
X_tab_test = tab_preprocessor.transform(test)
y_test = test[target_col].values

/usr/local/lib/python3.12/dist-packages/pytorch_widedeep/preprocessing/tab_preprocessor.py:364: UserWarning: Continuous columns will not be normalised
  warnings.warn("Continuous columns will not be normalised")
/usr/local/lib/python3.12/dist-packages/pytorch_widedeep/preprocessing/tab_preprocessor.py:364: UserWarning: Continuous columns will not be normalised
  warnings.warn("Continuous columns will not be normalised")
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.

In [5]:
# Create the TabMlp component
tab_mlp = TabMlp(
    column_idx=tab_preprocessor.column_idx,
    cat_embed_input=tab_preprocessor.cat_embed_input,
    continuous_cols=num_cols,
    mlp_hidden_dims=[250, 150], # 2 layers: 250 and 150 neurons
    mlp_activation="relu"
)

# Wrap in the WideDeep container (required by the library even if only using Deep)
model = WideDeep(deeptabular=tab_mlp)

In [6]:
# Initialize Trainer
trainer = Trainer(
    model,
    objective="mse",
    num_workers=0
)

# Train the model
trainer.fit(
    X_tab=X_tab_train,
    target=y_train,
    X_tab_val=X_tab_val,
    val_target=y_val,
    n_epochs=80,
    batch_size=64
)

epoch 80: 100%|██████████| 1001/1001 [00:10<00:00, 99.49it/s, loss=2e+9]


3.Report the test MSE and the test R2 value that you obtained.

In [8]:
# YOUR CODE & RESULT HERE

# 1. Generate predictions on the test set
# X_tab_test is the transformed test set from the TabPreprocessor
preds = trainer.predict(X_tab=X_tab_test)

# 2. Calculate the metrics
test_mse = mean_squared_error(y_test, preds)
test_r2 = r2_score(y_test, preds)

# 3. Report the values
print(f"Test MSE: {test_mse:,.4f}")
print(f"Test R2 Value: {test_r2:.4f}")

predict: 100%|██████████| 1128/1128 [00:03<00:00, 314.40it/s]


Test MSE: 11,912,049,639.3414
Test R2 Value: 0.5838


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
